# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*   **Unit of Analysis**: A **single content item (page) for a given client** (Grain: `client_id` + `content_id`). Every row represents one page.
*   **Time Window**: The data contains retrospectiveaggregations over a **90-day performance window** (clicks, impressions, positions, etc. are 90d aggregates prior to the data export of `2026-07-03`; daily performance trends end on `2026-06-30` because the freshest 3 days were cut off to prevent incomplete daily facts).

In [1]:
# Find repo root and load the dataset
import os, pandas as pd, numpy as np
for _ in range(5):
    if os.path.isdir("data/raw"): break
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working directory:", os.getcwd())
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Unique clients: {df['client_id'].nunique()} | Unique pages: {df['content_id'].nunique():,}")

Working directory: D:\Flyrank_internship\Ml_intership
Dataset shape: 30,000 rows x 44 columns
Unique clients: 32 | Unique pages: 30,000


## 2. Fields: feature / label / context / excluded

*   **Feature (Safe to model)**:
    *   *Numeric search signals*: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `ctr`, `avg_position`, `days_with_impressions`, `days_with_sessions`.
    *   *Numeric engagement signals*: `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.
    *   *Numeric freshness signals*: `content_age_days`, `days_since_last_update`.
    *   *Categorical metadata*: `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`.
*   **Label / Proxy**:
    *   `trend_direction` (from which our target label `is_declining_label = (trend_direction == 'down')` is derived).
*   **Context (Keys & Helpers)**:
    *   `client_id`, `content_id` (used for joins, validation splits, and grouping; never modeled directly).
*   **Excluded (Why)**:
    *   `trend_pct`: **Excluded due to direct leakage**. It is the exact percentage change used to calculate `trend_direction`, making it the answer in disguise.
    *   Product decisions/flags (like prioritizations/health scores): **Excluded** because they are app outputs; modeling them creates a circular logic loop.

In [2]:
# Verify that trend_pct is highly correlated with trend_direction and would leak the label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
mean_pct_by_trend = df.groupby("trend_direction")["trend_pct"].mean()
print("Mean trend_pct by trend direction:")
print(mean_pct_by_trend.to_string())
assert "trend_pct" not in ["content_id", "client_id"], "Self-check: trend_pct must be excluded from feature vectors!"

Mean trend_pct by trend direction:
trend_direction
down      -58.113830
flat             NaN
new              NaN
stable     -3.185944
up        190.673997


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify our data contract assumptions with three queries:
1.  **Grain Verification**: Ensure `content_id` has no duplicates (each row is unique).
2.  **Missing Value Patterns**: Check columns for null values to see if any signals are systematically missing.
3.  **Data Range Audit**: Check minimum, maximum, and average values for key metrics to ensure no corrupted ranges.

In [3]:
# 1. Grain test: verify zero duplicate content_id rows
duplicate_count = df.groupby("content_id").size().loc[lambda x: x > 1].shape[0]
print(f"Duplicate content_id count: {duplicate_count}")
assert duplicate_count == 0, "Duplicate content_id found!"

# 2. Missing Value check
missing_values = df.isnull().mean()
print("\nMissing values ratio per column:")
print(missing_values[missing_values > 0].round(4).to_string())

# 3. Signal Range check
print("\nKey metrics ranges:")
print(df[["impressions_90d", "ctr", "avg_position", "content_age_days"]].describe().round(2).to_string())

Duplicate content_id count: 0

Missing values ratio per column:
search_volume        0.0823
competition          0.0823
competition_level    0.0870
cpc                  0.0823
main_intent          0.0791
word_count           0.2566
char_count           0.2566
provider_used        0.7146
model_used           0.1911
word_count_tier      0.2566
char_count_tier      0.2566
scroll_rate          0.0042
trend_pct            0.1129

Key metrics ranges:
       impressions_90d       ctr  avg_position  content_age_days
count         30000.00  30000.00      30000.00          30000.00
mean           5200.37      0.51         16.34            256.17
std           16838.02      3.28         15.22            132.71
min               1.00      0.00          0.00             90.00
25%              81.00      0.00          6.20            132.00
50%             731.00      0.07         10.80            236.00
75%            3615.25      0.29         22.30            333.00
max          517715.00    100.0

## 4. Data limits

*   **Unbalanced History**: The GA4 and GSC setups started at different times for different clients. Low impressions or sessions on early pages could be due to missing analytics integration rather than true zero traffic.
*   **No Causal Proof**: Since this is observational data, we cannot establish causal relationships. For example, low CTR might be caused by competitors, shifting search intent, or poor meta titles — our model can only note the association, not the cause.
*   **Time-Window Overlaps**: In the starter CSV, features (e.g. 90-day impressions) and the label (`trend_direction`) are computed over the same time window, introducing leakage. For a clean capstone, the features window must end strictly before the target window begins.

In [4]:
# Check age distributions to show the unbalanced nature of page histories
print("Content Age distributions by percentiles:")
print(df["content_age_days"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())

Content Age distributions by percentiles:
0.10    104.0
0.25    132.0
0.50    236.0
0.75    333.0
0.90    463.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.